
# Experimentación: análisis de sentimientos

## 1. Objetivo

Evaluar modelos de procesamiento del lenguaje natural para identificar el sentimiento general expresado en noticias digitales.

Se consideran tres categorías de clasificación:

- **Positivo:** contenido con una valoración predominantemente favorable.
- **Negativo:** contenido con una valoración predominantemente desfavorable.
- **Neutral:** contenido informativo o descriptivo sin una valoración predominante.

La clasificación se realiza sobre el título y la descripción de las noticias, sin considerar inicialmente una temática o entidad específica.

## 2. Metodología

Se utilizarán modelos Transformer preentrenados para clasificación de sentimientos.

El procedimiento experimental comprende:

1. Recuperación de noticias candidatas desde PostgreSQL.
2. Construcción del texto de entrada.
3. Ejecución del modelo de clasificación.
4. Inspección de etiquetas y puntuaciones.
5. Evaluación de resultados mediante métricas de clasificación.

El procesamiento se realizará localmente, sin enviar los textos a servicios externos de inferencia.

Los resultados permitirán determinar la adecuación del modelo para el corpus de noticias y fundamentar su selección para el pipeline definitivo.


In [3]:
import pandas as pd
import torch
import psycopg
import os
from transformers import pipeline

In [4]:
# Configuración del experimento.
SENTIMENT_CONFIG = {
    "model_name": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    "max_length": 512,
    "batch_size": 8,
    "device": -1,
}

# Inicializar el clasificador de sentimientos.
sentiment_classifier = pipeline(
    task="text-classification",
    model=SENTIMENT_CONFIG["model_name"],
    tokenizer=SENTIMENT_CONFIG["model_name"],
    device=SENTIMENT_CONFIG["device"],
)

print("Modelo de análisis de sentimientos cargado correctamente.")


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

e:\Trabajo final UGR\communication-intelligence-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\enzor\.cache\huggingface\hub\models--cardiffnlp--twitter-xlm-roberta-base-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\Trabajo final UGR\communication-intelligence-assistant\.

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Modelo de análisis de sentimientos cargado correctamente.


In [5]:
def get_database_connection():
    """Crea una conexión con la base de datos PostgreSQL configurada.

    Returns:
        psycopg.Connection: Conexión activa a la base de datos PostgreSQL.
    """
    return psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

In [6]:

def get_candidate_news(connection, topic):
    """
    Recupera las noticias candidatas para el análisis
    de sentimientos de una temática configurada.

    Parameters
    ----------
    connection : psycopg.Connection
        Conexión activa a PostgreSQL.
    topic : str
        Identificador de la temática.

    Returns
    -------
    pandas.DataFrame
        Noticias candidatas con título y descripción.
    """
    query = """
        SELECT
            n.rss_news_id,
            n.title,
            n.description
        FROM rss_news AS n
        INNER JOIN nli_news AS nn
            ON n.rss_news_id = nn.rss_news_id
        WHERE nn.topic = %(topic)s
          AND (
              nn.keyword_match = TRUE
              OR nn.nli_recovered = TRUE
          )
        ORDER BY n.rss_news_id;
    """

    return pd.read_sql_query(
        query,
        connection,
        params={"topic": topic},
    )


# Recuperar noticias candidatas.
connection = get_database_connection()

try:
    sentiment_news = get_candidate_news(
        connection=connection,
        topic="inteligencia_artificial",
    )
finally:
    connection.close()

print(f"Noticias candidatas: {len(sentiment_news)}")


Noticias candidatas: 76


C:\Users\enzor\AppData\Local\Temp\ipykernel_30004\1078968999.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(


In [7]:

def build_sentiment_text(row):
    """
    Construye el texto de entrada a partir del
    título y la descripción de una noticia.
    """
    title = str(row["title"] or "").strip()
    description = str(row["description"] or "").strip()

    return f"{title}. {description}".strip()


sentiment_news["sentiment_text"] = sentiment_news.apply(
    build_sentiment_text,
    axis=1,
)


In [8]:

# Seleccionar una muestra reproducible.
sample_news = sentiment_news.sample(
    n=min(10, len(sentiment_news)),
    random_state=42,
).copy()

# Ejecutar la clasificación de sentimientos.
predictions = sentiment_classifier(
    sample_news["sentiment_text"].tolist(),
    truncation=True,
    max_length=SENTIMENT_CONFIG["max_length"],
    batch_size=SENTIMENT_CONFIG["batch_size"],
)

# Incorporar las predicciones al DataFrame.
sample_news["sentiment_label"] = [
    prediction["label"]
    for prediction in predictions
]

sample_news["sentiment_score"] = [
    prediction["score"]
    for prediction in predictions
]

display(
    sample_news[
        [
            "rss_news_id",
            "title",
            "sentiment_label",
            "sentiment_score",
        ]
    ].round(4)
)


,rss_news_id,title,sentiment_label,sentiment_score
4,13,Bill Gates pide reservar empleos solo para hum...,negative,0.4550
35,89,Cómo se verían Las Guerreras K-pop como person...,neutral,0.6369
10,24,Un robot humanoide supera el récord humano de ...,positive,0.7426
0,3,Bill Gates advierte sobre la IA: “será el mayo...,negative,0.8888
45,113,"Qué esconde el proyecto “Edward Warchocki”, el...",neutral,0.6528
47,115,El desarrollo global de reactores modulares es...,positive,0.4894
65,133,La historia de Sci-Hub: la biblioteca pirata q...,positive,0.5660
53,121,Redes sociales y menores de edad: crecen las r...,negative,0.7520
50,118,"Erling Haaland, el jugador noruego que la romp...",neutral,0.5234
28,71,Otro ejecutivo de OpenAI se va: la compañía pi...,negative,0.7692



## 3. Clasificación de sentimientos del corpus

### 3.1. Objetivo

Aplicar el modelo de clasificación de sentimientos sobre el conjunto completo de noticias candidatas, identificadas previamente mediante coincidencias léxicas y recuperación semántica.

El procesamiento permite asignar a cada noticia una etiqueta de sentimiento general:

- **Positivo:** predominio de expresiones favorables.
- **Negativo:** predominio de expresiones desfavorables.
- **Neutral:** ausencia de una valoración claramente predominante.

### 3.2. Metodología

Se utiliza el modelo multilingüe `cardiffnlp/twitter-xlm-roberta-base-sentiment`, basado en la arquitectura XLM-RoBERTa.

Para cada noticia se procesan el título y la descripción, obteniendo las puntuaciones correspondientes a las tres clases.

Se conserva tanto la etiqueta predicha como las puntuaciones individuales, permitiendo inspeccionar la distribución de resultados y analizar posibles ambigüedades.

Las puntuaciones representan las salidas normalizadas del clasificador y no deben interpretarse como probabilidades calibradas ni como medidas directas de intensidad emocional.

Esta etapa tiene carácter experimental. Los resultados no se almacenan todavía en PostgreSQL.


In [9]:

def classify_news_sentiment(news, classifier, config):
    """
    Clasifica el sentimiento general de las noticias.

    Parameters
    ----------
    news : pandas.DataFrame
        Noticias con la columna sentiment_text.
    classifier : transformers.Pipeline
        Modelo de clasificación de sentimientos.
    config : dict
        Configuración de inferencia.

    Returns
    -------
    pandas.DataFrame
        Noticias con etiqueta predicha y puntuaciones
        de las tres clases de sentimiento.
    """
    results = news.copy()

    if results.empty:
        raise ValueError("No hay noticias para clasificar.")

    predictions = classifier(
        results["sentiment_text"].tolist(),
        truncation=True,
        max_length=config["max_length"],
        batch_size=config["batch_size"],
        top_k=None,
    )

    # Normalizar las puntuaciones de cada clase.
    scores = [
        {
            item["label"].lower(): float(item["score"])
            for item in prediction
        }
        for prediction in predictions
    ]

    score_columns = [
        "negative",
        "neutral",
        "positive",
    ]

    score_results = pd.DataFrame(
        scores,
        index=results.index,
    )[score_columns]

    score_results = score_results.rename(
        columns={
            "negative": "negative_score",
            "neutral": "neutral_score",
            "positive": "positive_score",
        }
    )

    results = pd.concat(
        [results, score_results],
        axis=1,
    )

    # Seleccionar la clase con mayor puntuación.
    results["sentiment_label"] = (
        score_results.idxmax(axis=1)
        .str.replace("_score", "", regex=False)
    )

    results["sentiment_score"] = score_results.max(axis=1)

    return results


In [10]:

# Ejecutar la clasificación sobre todas las noticias candidatas.
sentiment_results = classify_news_sentiment(
    news=sentiment_news,
    classifier=sentiment_classifier,
    config=SENTIMENT_CONFIG,
)

print(f"Noticias procesadas: {len(sentiment_results)}")

display(
    sentiment_results[
        [
            "rss_news_id",
            "title",
            "sentiment_label",
            "negative_score",
            "neutral_score",
            "positive_score",
        ]
    ].head(10).round(4)
)


Noticias procesadas: 76


,rss_news_id,title,sentiment_label,negative_score,neutral_score,positive_score
0,3,Bill Gates advierte sobre la IA: “será el mayo...,negative,0.8888,0.0947,0.0165
1,7,Time reveló las 100 personalidades más influye...,neutral,0.0432,0.5115,0.4454
2,8,Un informe reveló cómo 700 agentes de IA de Op...,negative,0.8463,0.1378,0.0158
3,12,El costo de Bitcoin para este jueves,neutral,0.0241,0.6844,0.2915
4,13,Bill Gates pide reservar empleos solo para hum...,negative,0.4550,0.4021,0.1429
5,16,Cada vez más países ven publicidad en ChatGPT:...,neutral,0.1970,0.7325,0.0705
6,17,Los centros de datos de Estados Unidos ya cons...,neutral,0.0541,0.4951,0.4508
7,19,Propinas para robots: la polémica detrás de la...,negative,0.7631,0.2185,0.0184
8,21,Amazon se une a Nvidia y tendrá dos millones d...,positive,0.0152,0.3497,0.6351
9,23,Por qué Londres está frenando la llegada de lo...,negative,0.8810,0.1040,0.0150



## 4. Análisis exploratorio de sentimientos

Se realiza una inspección descriptiva de los resultados para comprender el comportamiento del clasificador sobre el corpus de noticias.

Se analizan tres aspectos:

1. Distribución de noticias según el sentimiento predicho.
2. Estadísticas descriptivas de las puntuaciones del modelo.
3. Identificación de noticias con predicciones poco concluyentes.

Este análisis permite detectar posibles sesgos, dificultades de clasificación y casos que requieren revisión.

Los resultados no constituyen todavía una evaluación del rendimiento, ya que no se dispone de etiquetas de referencia verificadas para esta tarea.


In [11]:

# Calcular la distribución de sentimientos.
sentiment_distribution = (
    sentiment_results["sentiment_label"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="news_count")
)

sentiment_distribution["percentage"] = (
    sentiment_distribution["news_count"]
    / len(sentiment_results)
    * 100
)

display(sentiment_distribution.round(2))


,sentiment,news_count,percentage
0,neutral,41,53.95
1,positive,20,26.32
2,negative,15,19.74


In [12]:

# Analizar las puntuaciones obtenidas por el modelo.
score_columns = [
    "negative_score",
    "neutral_score",
    "positive_score",
]

display(
    sentiment_results[score_columns]
    .describe()
    .round(4)
)


,negative_score,neutral_score,positive_score
count,76.0000,76.0000,76.0000
mean,0.2173,0.4854,0.2973
std,0.2792,0.1940,0.2139
min,0.0151,0.0947,0.0117
25%,0.0367,0.3486,0.0878
50%,0.0713,0.4860,0.2736
75%,0.2292,0.6189,0.4789
max,0.8888,0.8595,0.7426



### 4.1. Análisis de ambigüedad

Para identificar predicciones potencialmente ambiguas se calcula el margen entre las dos clases con mayor puntuación.

Un margen reducido indica que el clasificador asigna puntuaciones similares a dos categorías, mientras que un margen elevado indica una mayor separación entre ellas.

Este indicador se utiliza exclusivamente para inspección exploratoria y no constituye una medida calibrada de incertidumbre.


In [13]:

import numpy as np


# Ordenar las puntuaciones de cada noticia.
sorted_scores = np.sort(
    sentiment_results[score_columns].to_numpy(),
    axis=1,
)

# Calcular el margen entre las dos clases principales.
sentiment_results["prediction_margin"] = (
    sorted_scores[:, -1] - sorted_scores[:, -2]
)

# Identificar las predicciones con menor margen.
ambiguous_news = sentiment_results.nsmallest(
    15,
    "prediction_margin",
)

display(
    ambiguous_news[
        [
            "rss_news_id",
            "title",
            "sentiment_label",
            "negative_score",
            "neutral_score",
            "positive_score",
            "prediction_margin",
        ]
    ].round(4)
)


,rss_news_id,title,sentiment_label,negative_score,neutral_score,positive_score,prediction_margin
54,122,¿De qué manera las universidades promueven el ...,neutral,0.0415,0.4798,0.4787,0.0011
63,131,Sabiduría colectiva: un estudio muestra la efi...,positive,0.0305,0.4811,0.4884,0.0074
41,98,Gen Z y Alfa representan a la generación que f...,positive,0.0726,0.4599,0.4676,0.0077
51,119,La UBA lanza un chatbot entrenado exclusivamen...,neutral,0.0301,0.4904,0.4795,0.0109
13,30,Cómo obtener Google AI Plus gratis para estudi...,positive,0.0193,0.4816,0.4992,0.0176
30,80,Bill Gates alerta al mundo: “la transición a l...,neutral,0.4038,0.4222,0.1740,0.0184
73,166,El futuro de la IA y su impacto en el conocimi...,positive,0.0809,0.4485,0.4706,0.0221
6,17,Los centros de datos de Estados Unidos ya cons...,neutral,0.0541,0.4951,0.4508,0.0442
47,115,El desarrollo global de reactores modulares es...,positive,0.0684,0.4422,0.4894,0.0471
4,13,Bill Gates pide reservar empleos solo para hum...,negative,0.4550,0.4021,0.1429,0.0529



## 5. Evaluación del modelo de análisis de sentimientos

### 5.1. Construcción del conjunto de referencia

Para evaluar el rendimiento del modelo se construirá un conjunto de referencia mediante el etiquetado manual de las noticias.

Se utilizarán tres categorías:

- **positive:** el texto presenta una valoración predominantemente favorable.
- **negative:** el texto presenta una valoración predominantemente desfavorable.
- **neutral:** el texto es principalmente informativo, descriptivo o no expresa una valoración predominante.

La anotación se realizará considerando exclusivamente el título y la descripción, que constituyen las entradas del modelo.

Las etiquetas de referencia se asignarán sin consultar las predicciones automáticas para evitar introducir sesgos en la evaluación.


In [15]:

import pandas as pd


synthetic_cases = [
    # Casos positivos
    (1, "Nueva IA mejora los diagnósticos médicos",
     "Un sistema de inteligencia artificial logró detectar enfermedades con mayor precisión y reducir los tiempos de diagnóstico.",
     "positive", "claro"),
    (2, "Avance histórico en energías renovables",
     "Investigadores celebran una tecnología que reduce significativamente las emisiones contaminantes.",
     "positive", "claro"),
    (3, "Estudiantes desarrollan una herramienta inclusiva",
     "La aplicación recibió elogios por facilitar el acceso a la educación de personas con discapacidad.",
     "positive", "claro"),
    (4, "Empresa tecnológica alcanza resultados excelentes",
     "La compañía anunció un crecimiento récord y destacó la satisfacción de sus clientes.",
     "positive", "claro"),
    (5, "Nuevo tratamiento ofrece resultados prometedores",
     "Los primeros ensayos muestran mejoras importantes y generan entusiasmo entre los investigadores.",
     "positive", "claro"),

    # Casos negativos
    (6, "Falla de IA provoca errores médicos",
     "Un sistema automatizado produjo diagnósticos incorrectos y generó preocupación por la seguridad de los pacientes.",
     "negative", "claro"),
    (7, "Filtración masiva de datos personales",
     "Una empresa sufrió un grave incidente de seguridad que expuso información confidencial de miles de usuarios.",
     "negative", "claro"),
    (8, "Crisis económica afecta a miles de familias",
     "El aumento del desempleo y la inflación agrava las dificultades económicas de la población.",
     "negative", "claro"),
    (9, "Investigación revela discriminación algorítmica",
     "Una auditoría detectó sesgos graves que perjudican a determinados grupos de personas.",
     "negative", "claro"),
    (10, "Empresa cancela proyecto tecnológico",
     "El desarrollo fue suspendido después de múltiples fallas y pérdidas económicas.",
     "negative", "claro"),

    # Casos neutrales
    (11, "Gobierno presenta informe sobre inteligencia artificial",
     "El documento describe las iniciativas de inteligencia artificial desarrolladas durante el último año.",
     "neutral", "claro"),
    (12, "Universidad anuncia conferencia tecnológica",
     "El encuentro se realizará el próximo viernes y contará con especialistas de distintas instituciones.",
     "neutral", "claro"),
    (13, "Empresa publica actualización de software",
     "La nueva versión estará disponible a partir del lunes para los usuarios registrados.",
     "neutral", "claro"),
    (14, "Investigadores presentan resultados de un estudio",
     "El trabajo analiza el uso de herramientas digitales en instituciones educativas.",
     "neutral", "claro"),
    (15, "Se publica un nuevo informe sobre empleo",
     "El documento reúne estadísticas laborales correspondientes al último trimestre.",
     "neutral", "claro"),

    # Casos ambiguos
    (16, "La IA transforma el mercado laboral",
     "La automatización crea nuevas oportunidades profesionales, pero también genera preocupación por la pérdida de empleos.",
     "neutral", "mixto"),
    (17, "Nueva tecnología obtiene resultados contradictorios",
     "El sistema mejora la productividad, aunque presenta fallas que afectan su confiabilidad.",
     "neutral", "mixto"),
    (18, "Empresa anuncia crecimiento y despidos",
     "La compañía registró ganancias récord mientras confirmó la reducción de su plantilla.",
     "neutral", "mixto"),

    # Negación
    (19, "La nueva IA no presenta fallas de seguridad",
     "Los investigadores confirmaron que las pruebas no detectaron vulnerabilidades importantes.",
     "positive", "negación"),
    (20, "El sistema no logró mejorar los resultados",
     "La herramienta presentó dificultades y no alcanzó los objetivos establecidos.",
     "negative", "negación"),

    # Ironía y sarcasmo
    (21, "Excelente avance: el sistema dejó de funcionar",
     "La nueva plataforma falló durante su presentación y provocó múltiples interrupciones.",
     "negative", "ironía"),
    (22, "Otro éxito tecnológico: miles de usuarios sin servicio",
     "Una actualización defectuosa dejó inaccesible la plataforma durante varias horas.",
     "negative", "ironía"),

    # Titulares sensacionalistas
    (23, "La inteligencia artificial cambiará TODO para siempre",
     "Un informe analiza posibles aplicaciones futuras de sistemas de inteligencia artificial.",
     "neutral", "sensacionalismo"),
    (24, "Impactante descubrimiento revoluciona la ciencia",
     "Investigadores publicaron un estudio sobre nuevos métodos de procesamiento de datos.",
     "neutral", "sensacionalismo"),
]

synthetic_df = pd.DataFrame(
    synthetic_cases,
    columns=[
        "case_id",
        "title",
        "description",
        "expected_sentiment",
        "case_type",
    ],
)

synthetic_df["sentiment_text"] = (
    synthetic_df["title"] + ". " + synthetic_df["description"]
)

display(synthetic_df)


,case_id,title,description,expected_sentiment,case_type,sentiment_text
0,1,Nueva IA mejora los diagnósticos médicos,Un sistema de inteligencia artificial logró de...,positive,claro,Nueva IA mejora los diagnósticos médicos. Un s...
1,2,Avance histórico en energías renovables,Investigadores celebran una tecnología que red...,positive,claro,Avance histórico en energías renovables. Inves...
2,3,Estudiantes desarrollan una herramienta inclusiva,La aplicación recibió elogios por facilitar el...,positive,claro,Estudiantes desarrollan una herramienta inclus...
3,4,Empresa tecnológica alcanza resultados excelentes,La compañía anunció un crecimiento récord y de...,positive,claro,Empresa tecnológica alcanza resultados excelen...
4,5,Nuevo tratamiento ofrece resultados prometedores,Los primeros ensayos muestran mejoras importan...,positive,claro,Nuevo tratamiento ofrece resultados prometedor...
5,6,Falla de IA provoca errores médicos,Un sistema automatizado produjo diagnósticos i...,negative,claro,Falla de IA provoca errores médicos. Un sistem...
6,7,Filtración masiva de datos personales,Una empresa sufrió un grave incidente de segur...,negative,claro,Filtración masiva de datos personales. Una emp...
7,8,Crisis económica afecta a miles de familias,El aumento del desempleo y la inflación agrava...,negative,claro,Crisis económica afecta a miles de familias. E...
8,9,Investigación revela discriminación algorítmica,Una auditoría detectó sesgos graves que perjud...,negative,claro,Investigación revela discriminación algorítmic...
9,10,Empresa cancela proyecto tecnológico,El desarrollo fue suspendido después de múltip...,negative,claro,Empresa cancela proyecto tecnológico. El desar...


In [16]:

from sklearn.metrics import classification_report, confusion_matrix

synthetic_results = classify_news_sentiment(
    news=synthetic_df,
    classifier=sentiment_classifier,
    config=SENTIMENT_CONFIG,
)

labels = ["negative", "neutral", "positive"]

print(
    classification_report(
        synthetic_results["expected_sentiment"],
        synthetic_results["sentiment_label"],
        labels=labels,
        zero_division=0,
    )
)

display(
    synthetic_results[
        [
            "case_id",
            "case_type",
            "expected_sentiment",
            "sentiment_label",
            "sentiment_score",
        ]
    ].round(4)
)


              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         8
     neutral       0.88      0.70      0.78        10
    positive       0.62      0.83      0.71         6

    accuracy                           0.83        24
   macro avg       0.83      0.84      0.83        24
weighted avg       0.85      0.83      0.84        24



,case_id,case_type,expected_sentiment,sentiment_label,sentiment_score
0,1,claro,positive,positive,0.6793
1,2,claro,positive,positive,0.8324
2,3,claro,positive,positive,0.8417
3,4,claro,positive,positive,0.9278
4,5,claro,positive,positive,0.9154
5,6,claro,negative,negative,0.8547
6,7,claro,negative,negative,0.8063
7,8,claro,negative,negative,0.8509
8,9,claro,negative,negative,0.8414
9,10,claro,negative,negative,0.7991


In [20]:

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


# 1. Cargar las etiquetas revisadas manualmente.
manual_evaluation = pd.read_csv(
    "sentiment_evaluation_suggested.csv",
    encoding="utf-8-sig",
)

manual_evaluation["manual_sentiment"] = (
    manual_evaluation["suggested_sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("*", "", regex=False)
)
# 2. Validar las etiquetas.
labels = ["negative", "neutral", "positive"]

invalid_rows = manual_evaluation[
    ~manual_evaluation["manual_sentiment"].isin(labels)
]

if not invalid_rows.empty:
    raise ValueError(
        f"Existen {len(invalid_rows)} noticias sin "
        "una etiqueta manual válida."
    )

if manual_evaluation["rss_news_id"].duplicated().any():
    raise ValueError("Existen identificadores duplicados.")

# 3. Recuperar las predicciones originales del modelo.
evaluation_results = manual_evaluation.merge(
    sentiment_results[
        ["rss_news_id", "sentiment_label"]
    ],
    on="rss_news_id",
    how="inner",
    validate="one_to_one",
)

if len(evaluation_results) != len(sentiment_results):
    raise ValueError(
        "No coinciden las noticias etiquetadas "
        "con las predicciones originales."
    )

# 4. Calcular las métricas.
y_true = evaluation_results["manual_sentiment"]
y_pred = evaluation_results["sentiment_label"]

print(f"Noticias evaluadas: {len(evaluation_results)}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

print("\nReporte de clasificación:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )
)

# 5. Matriz de confusión.
confusion_df = pd.DataFrame(
    confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
    ),
    index=[f"Real: {label}" for label in labels],
    columns=[f"Predicho: {label}" for label in labels],
)

display(confusion_df)

# 6. Identificar errores de clasificación.
classification_errors = evaluation_results[
    evaluation_results["manual_sentiment"]
    != evaluation_results["sentiment_label"]
].copy()

print(
    f"\nNoticias clasificadas incorrectamente: "
    f"{len(classification_errors)}"
)

display(
    classification_errors[
        [
            "rss_news_id",
            "title",
            "manual_sentiment",
            "sentiment_label",
        ]
    ]
)


Noticias evaluadas: 76
Accuracy: 0.6053

Reporte de clasificación:
              precision    recall  f1-score   support

    negative       0.73      0.69      0.71        16
     neutral       0.68      0.62      0.65        45
    positive       0.35      0.47      0.40        15

    accuracy                           0.61        76
   macro avg       0.59      0.59      0.59        76
weighted avg       0.63      0.61      0.61        76



,Predicho: negative,Predicho: neutral,Predicho: positive
Real: negative,11,5,0
Real: neutral,4,28,13
Real: positive,0,8,7



Noticias clasificadas incorrectamente: 30


,rss_news_id,title,manual_sentiment,sentiment_label
6,17,Los centros de datos de Estados Unidos ya cons...,negative,neutral
8,21,Amazon se une a Nvidia y tendrá dos millones d...,neutral,positive
11,26,Google lanzó Gemini 3.5 Transcribe: pasa más d...,positive,neutral
12,27,IA en el quirófano: así fue la primera operaci...,positive,neutral
16,36,La puerta es el mejor lugar para guardar los l...,neutral,positive
17,39,"La IA avanza en Perú, pero las empresas enfren...",neutral,positive
18,42,América Latina en el centro de la estrategia t...,neutral,positive
21,49,"Así funciona ProPILOT, un asistente de IA que ...",positive,neutral
23,56,Temblor en Colombia: aplicaciones y opciones p...,positive,neutral
24,57,Elon Musk construirá en Luisiana un segundo St...,neutral,positive
